In [ ]:
import sys
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print(sys.executable)

# 1. Загрузка модели FRIDA
print("Загрузка модели FRIDA...")
model = SentenceTransformer('ai-forever/FRIDA')
print("Модель загружена.")

# Префикс FRIDA. Согласно карточке модели (ai-forever/FRIDA), для симметричных
# задач оценки сходства (STS, поиск парафразов) используется префикс
# "paraphrase: ". Без префикса вход подаётся не в том формате, на котором
# модель обучалась, и эмбеддинги получаются хуже.
FRIDA_PROMPT_NAME = "paraphrase"
FRIDA_PROMPT_FALLBACK = "paraphrase: "


# 2. Кодирование текстов с кэшированием и батчингом
_embedding_cache: dict = {}


def encode_texts(texts: list) -> dict:
    """Кодирует список текстов в эмбеддинги одним батчем с применением
    префикса FRIDA. Результаты кэшируются. Возвращает словарь text -> embedding."""
    to_encode = [t for t in set(texts) if t and t not in _embedding_cache]
    if to_encode:
        try:
            embs = model.encode(
                to_encode, prompt_name=FRIDA_PROMPT_NAME,
                batch_size=32, show_progress_bar=False,
            )
        except (TypeError, ValueError):
            embs = model.encode(
                [FRIDA_PROMPT_FALLBACK + t for t in to_encode],
                batch_size=32, show_progress_bar=False,
            )
        for t, e in zip(to_encode, embs):
            _embedding_cache[t] = e
    return {t: _embedding_cache[t] for t in texts if t}


def get_emb(text):
    """Эмбеддинг одного текста (или None для пустого/NaN)."""
    if pd.isna(text) or str(text) == "":
        return None
    return _embedding_cache.get(str(text))


# 3. Оценка семантической близости (вариант Б)
def semantic_evaluation(golden_df, pred_df):
    """Оценивает семантическую близость извлечённого текста.

    Сопоставление выполняется ОТДЕЛЬНО для аспектных терминов и ОТДЕЛЬНО
    для мнений, на уровне отдельных компонентов (не квадруплетов целиком).
    Аспектная категория и тональность в сопоставлении не участвуют.

    Для каждого отзыва предсказанные и эталонные фрагменты сопоставляются
    как задача оптимального паросочетания по косинусной близости их
    эмбеддингов. Косинусная близость каждой сопоставленной пары идёт в
    общий пул, по которому считается среднее.

    Возвращает словарь со средними значениями для терминов и мнений.
    """
    golden_df = golden_df.copy()
    pred_df = pred_df.copy()
    golden_df['review_id'] = golden_df['review_id'].astype(str)
    pred_df['review_id'] = pred_df['review_id'].astype(str)

    golden_grouped = golden_df.groupby('review_id')
    pred_grouped = pred_df.groupby('review_id')

    all_review_ids = set(golden_grouped.groups) | set(pred_grouped.groups)
    print(f"Найдено {len(all_review_ids)} review_id для сравнения.")

    # предварительное кодирование всех фрагментов одним проходом
    all_texts = []
    for col in ['aspect_term', 'opinion']:
        for df in (golden_df, pred_df):
            all_texts += [str(x) for x in df[col].dropna() if str(x) != ""]
    print(f"Кодирование {len(set(all_texts))} уникальных фрагментов...")
    encode_texts(all_texts)

    term_sims = []   # косинусные близости всех сопоставленных пар терминов
    op_sims = []     # косинусные близости всех сопоставленных пар мнений

    def match_component(gold_vals, pred_vals, sink):
        """Сопоставляет два списка фрагментов одного компонента оптимальным
        паросочетанием по косинусной близости; близости пар идут в sink."""
        gold_embs = [get_emb(v) for v in gold_vals]
        pred_embs = [get_emb(v) for v in pred_vals]
        gold_embs = [e for e in gold_embs if e is not None]
        pred_embs = [e for e in pred_embs if e is not None]

        if not gold_embs or not pred_embs:
            return

        sim = cosine_similarity(np.array(pred_embs), np.array(gold_embs))
        row_ind, col_ind = linear_sum_assignment(-sim)
        for i, j in zip(row_ind, col_ind):
            sink.append(float(sim[i, j]))

    for rid in tqdm(all_review_ids, desc="Обработка отзывов"):
        gold_rows = (golden_grouped.get_group(rid)
                     if rid in golden_grouped.groups else pd.DataFrame())
        pred_rows = (pred_grouped.get_group(rid)
                     if rid in pred_grouped.groups else pd.DataFrame())

        for col, sink in [('aspect_term', term_sims), ('opinion', op_sims)]:
            gold_vals = list(gold_rows[col]) if not gold_rows.empty else []
            pred_vals = list(pred_rows[col]) if not pred_rows.empty else []
            match_component(gold_vals, pred_vals, sink)

    summary = {
        'matched_term_pairs': len(term_sims),
        'matched_opinion_pairs': len(op_sims),
        'aspect_term_similarity_mean':
            float(np.mean(term_sims)) if term_sims else 0.0,
        'opinion_similarity_mean':
            float(np.mean(op_sims)) if op_sims else 0.0,
    }
    return summary


# 4. Запуск
if __name__ == "__main__":
    golden_df = pd.read_csv('golden_dataset.csv')
    pred_df = pd.read_csv('predictions_dataset_gpt.csv')

    summary = semantic_evaluation(golden_df, pred_df)

    print("\n" + "=" * 60)
    print("       СРЕДНЯЯ СЕМАНТИЧЕСКАЯ БЛИЗОСТЬ (модель FRIDA)")
    print("=" * 60)
    print(f"Сопоставлено пар аспектных терминов : {summary['matched_term_pairs']}")
    print(f"Сопоставлено пар мнений            : {summary['matched_opinion_pairs']}")
    print(f"Средняя близость аспектных терминов : {summary['aspect_term_similarity_mean']:.4f}")
    print(f"Средняя близость мнений            : {summary['opinion_similarity_mean']:.4f}")